In [1]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM


model_path = os.path.expanduser('~/data/models/Janus-Pro-7B-language')
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

tokenizer

/home/camus/work/deep-starry/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
2025-03-08 15:45:48.368672: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN

LlamaTokenizerFast(name_or_path='/home/camus/data/models/Janus-Pro-7B-language', vocab_size=100000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	100000: AddedToken("<｜begin▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	100001: AddedToken("<｜end▁of▁sentence｜>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	100002: AddedToken("ø", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	100003: AddedToken("ö", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	100004: AddedToken("ú", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	100005: AddedToken("ÿ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False)

In [2]:
import torch

model.to(torch.bfloat16).to('cuda')

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(102400, 4096)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (n

In [4]:
input_ids = tokenizer.encode("Hello, my dog is cute", return_tensors="pt").to('cuda')
output_ids = model.generate(input_ids, max_length=100, num_return_sequences=1)
output_ids

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


tensor([[100000,  17464,     11,    601,   5025,    317,  15943,    285,    245,
           2603,    280,    829,     13,  17464,      0,    809,      6,     82,
           1228,    276,   4704,    344,    340,    463,    245,   5025,    344,
            317,  15943,    285,    829,     13,  45117,    418,   7289,  20935,
            285,    481,   3792,    558,   1266,   8810,    285,  14036,    276,
            769,   6110,     13, 100001, 100000,      6,     82,    245,   1228,
           1143,    276,   1330,    254,   1492,     13, 100001, 100000, 100001,
         100000,   5661,     11,    359,    317,     13,  45117,    481,    330,
            245,   1228,   3130,    280,   5186,    285,  69510,     11,    285,
            657,    481,    839,   1345,    450,   4152,   5467,    285,  12662,
             13, 100001, 100000, 100001, 100000, 100001, 100000,     40,      6,
             76]], device='cuda:0')

In [5]:
tokenizer.decode(output_ids[0], skip_special_tokens=True)

"Hello, my dog is cute and a lot of fun.Hello! It's great to hear that you have a dog that is cute and fun. Dogs are wonderful companions and can bring so much joy and happiness to our lives.'s a great way to start the day.Yes, it is. Dogs can be a great source of comfort and companionship, and they can also help us stay active and engaged.I'm"

In [6]:
import dataclasses
from enum import IntEnum, auto
from typing import Dict, List


class SeparatorStyle(IntEnum):
    """Separator styles."""

    ADD_COLON_SINGLE = auto()
    ADD_COLON_TWO = auto()
    ADD_COLON_SPACE_SINGLE = auto()
    NO_COLON_SINGLE = auto()
    NO_COLON_TWO = auto()
    ADD_NEW_LINE_SINGLE = auto()
    LLAMA2 = auto()
    CHATGLM = auto()
    CHATML = auto()
    CHATINTERN = auto()
    DOLLY = auto()
    RWKV = auto()
    PHOENIX = auto()
    ROBIN = auto()
    DeepSeek = auto()
    PLAIN = auto()
    ALIGNMENT = auto()


@dataclasses.dataclass
class Conversation:
    """A class that manages prompt templates and keeps all conversation history."""

    # The name of this template
    name: str
    # The template of the system prompt
    system_template: str = "{system_message}"
    # The system message
    system_message: str = ""
    # The names of two roles
    roles: List[str] = (("USER", "ASSISTANT"),)
    # All messages. Each item is (role, message).
    messages: List[List[str]] = ()
    # The number of few shot examples
    offset: int = 0
    # The separator style and configurations
    sep_style: SeparatorStyle = SeparatorStyle.ADD_COLON_SINGLE
    sep: str = "\n"
    sep2: str = None
    # Stop criteria (the default one is EOS token)
    stop_str: str = None
    # Stops generation if meeting any token in this list
    stop_token_ids: List[int] = None

    def get_prompt(self) -> str:
        """Get the prompt for generation."""
        system_prompt = self.system_template.format(system_message=self.system_message)

        if self.sep_style == SeparatorStyle.DeepSeek:
            seps = [self.sep, self.sep2]
            if system_prompt == "" or system_prompt is None:
                ret = ""
            else:
                ret = system_prompt + seps[0]
            for i, (role, message) in enumerate(self.messages):
                if message:
                    ret += role + ": " + message + seps[i % 2]
                else:
                    ret += role + ":"
            return ret
        elif self.sep_style == SeparatorStyle.LLAMA2:
            seps = [self.sep, self.sep2]
            if self.system_message:
                ret = system_prompt
            else:
                ret = "[INST] "
            for i, (role, message) in enumerate(self.messages):
                tag = self.roles[i % 2]
                if message:
                    if type(message) is tuple:  # multimodal message
                        message, _ = message
                    if i == 0:
                        ret += message + " "
                    else:
                        ret += tag + " " + message + seps[i % 2]
                else:
                    ret += tag
            return ret
        elif self.sep_style == SeparatorStyle.PLAIN:
            seps = [self.sep, self.sep2]
            ret = ""
            for i, (role, message) in enumerate(self.messages):
                if message:
                    if type(message) is tuple:
                        message, _, _ = message
                    if i % 2 == 0:
                        ret += message + seps[i % 2]
                    else:
                        ret += message + seps[i % 2]
                else:
                    ret += ""
            return ret
        elif self.sep_style == SeparatorStyle.ALIGNMENT:
            seps = [self.sep, self.sep2]
            ret = ""
            for i, (role, message) in enumerate(self.messages):
                if message:
                    if type(message) is tuple:
                        message, _, _ = message
                    if i % 2 == 0:
                        ret += "<image>\n" + seps[i % 2]
                    else:
                        ret += message + seps[i % 2]
                else:
                    ret += ""
            return ret
        else:
            raise ValueError(f"Invalid style: {self.sep_style}")

    def get_prompt_for_current_round(self, content=None):
        """Get current round formatted question prompt during sft training"""
        if self.sep_style == SeparatorStyle.PLAIN:
            formatted_question = "<image>\n"
        elif self.sep_style == SeparatorStyle.DeepSeek:
            formatted_question = (
                f"{self.roles[0]}: " + content.strip() + self.sep + f"{self.roles[1]}:"
            )
        else:
            raise ValueError(f"Unsupported sep_style: {self.sep_style}")
        return formatted_question

    def set_system_message(self, system_message: str):
        """Set the system message."""
        self.system_message = system_message

    def append_message(self, role: str, message: str):
        """Append a new message."""
        self.messages.append([role, message])

    def reset_message(self):
        """Reset a new message."""
        self.messages = []

    def update_last_message(self, message: str):
        """Update the last output.

        The last message is typically set to be None when constructing the prompt,
        so we need to update it in-place after getting the response from a model.
        """
        self.messages[-1][1] = message

    def to_gradio_chatbot(self):
        """Convert the conversation to gradio chatbot format."""
        ret = []
        for i, (role, msg) in enumerate(self.messages[self.offset :]):
            if i % 2 == 0:
                ret.append([msg, None])
            else:
                ret[-1][-1] = msg
        return ret

    def to_openai_api_messages(self):
        """Convert the conversation to OpenAI chat completion format."""
        system_prompt = self.system_template.format(system_message=self.system_message)
        ret = [{"role": "system", "content": system_prompt}]

        for i, (_, msg) in enumerate(self.messages[self.offset :]):
            if i % 2 == 0:
                ret.append({"role": "user", "content": msg})
            else:
                if msg is not None:
                    ret.append({"role": "assistant", "content": msg})
        return ret

    def copy(self):
        return Conversation(
            name=self.name,
            system_template=self.system_template,
            system_message=self.system_message,
            roles=self.roles,
            messages=[[x, y] for x, y in self.messages],
            offset=self.offset,
            sep_style=self.sep_style,
            sep=self.sep,
            sep2=self.sep2,
            stop_str=self.stop_str,
            stop_token_ids=self.stop_token_ids,
        )

    def dict(self):
        return {
            "template_name": self.name,
            "system_message": self.system_message,
            "roles": self.roles,
            "messages": self.messages,
            "offset": self.offset,
        }


In [7]:
conv = Conversation(
        name="deepseek",
        system_template="{system_message}",
        # system_message="You are a helpful assistant. Please answer truthfully and write out your "
        # "thinking step by step to be sure you get the right answer.",
        system_message="",
        roles=("<|User|>", "<|Assistant|>"),
        messages=[],
        offset=0,
        sep_style=SeparatorStyle.DeepSeek,
        sep="\n\n",
        sep2="<｜end▁of▁sentence｜>",
        stop_token_ids=[100001],
        stop_str=["<|User|>", "<｜end▁of▁sentence｜>"]
    )
conv

Conversation(name='deepseek', system_template='{system_message}', system_message='', roles=('<|User|>', '<|Assistant|>'), messages=[], offset=0, sep_style=<SeparatorStyle.DeepSeek: 15>, sep='\n\n', sep2='<｜end▁of▁sentence｜>', stop_str=['<|User|>', '<｜end▁of▁sentence｜>'], stop_token_ids=[100001])

In [16]:
messages = [
	dict(
		role='<|User|>',
		content='Hello! What can you do for me today?',
	),
	dict(
		role='<|Assistant|>',
		content='',
	),
]

conv.set_system_message('You are a helpful assistant.')
for message in messages:
    conv.append_message(message["role"], message["content"].strip())
sft_prompt = conv.get_prompt().strip()
print(sft_prompt)

You are a helpful assistant.

<|User|>: Hello!

<|Assistant|>:<|User|>: Hello! What can you do for me today?

<|Assistant|>:


In [19]:
input_ids = tokenizer.encode(sft_prompt, return_tensors="pt").to('cuda')
output_ids = model.generate(input_ids, max_length=100, num_return_sequences=1,  eos_token_id=tokenizer.eos_token_id)
output_ids

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


tensor([[100000,   2054,    418,    245,   9394,  20308,     13,    185,    185,
         100601,     25,  37727,      0,    185,    185, 100602,     25, 100601,
             25,  37727,      0,   2461,    481,    340,    536,    327,    525,
           3571,     30,    185,    185, 100602,     25,  17464,      0,    304,
            481,   1345,    340,    366,    245,   6265,    280,   9224,     13,
           2461,    744,    340,    837,    276,   1006,    410,    536,     30,
         100001]], device='cuda:0')

In [20]:
print(tokenizer.decode(output_ids[0]))

<｜begin▁of▁sentence｜>You are a helpful assistant.

<|User|>: Hello!

<|Assistant|>:<|User|>: Hello! What can you do for me today?

<|Assistant|>:Hello! I can help you with a variety of tasks. What would you like to know or do?<｜end▁of▁sentence｜>


---
## tokenizer

In [1]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM


model_path = os.path.expanduser('~/data/models/Janus-Pro-7B-language')
tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


LlamaTokenizerFast(name_or_path='/home/camus/data/models/Janus-Pro-7B-language', vocab_size=100000, model_max_length=16384, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<｜begin▁of▁sentence｜>', 'eos_token': '<｜end▁of▁sentence｜>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	38: AddedToken("G", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	54: AddedToken("W", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	64: AddedToken("a", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	65: AddedToken("b", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	66: AddedToken("c", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	67: AddedToken("d", rstrip=False, lstrip=False, single_word=True, normalized=False, special=False),
	68: AddedToken("e", rstrip=False, lstrip=False, single_wo

In [ ]:
tokenizer.encode('BOM K1 TN3 TD4 a D4 G b EOM', add_special_tokens=False)

[100605,
 207,
 100612,
 207,
 100626,
 207,
 100637,
 207,
 64,
 207,
 100647,
 207,
 38,
 207,
 65,
 207,
 100606]

In [3]:
tokenizer.encode('world', add_special_tokens=False)

[11123]

In [4]:
tokenizer.decode([207])

2025-03-20 10:26:14.982379: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-20 10:26:14.982417: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-20 10:26:14.983630: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-20 10:26:14.990270: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-20 10:26:16.017381: W tensorflow/compiler/tf2

' '